# Day 4 — Activations, Gradients, BatchNorm (makemore Part 3, video 4, 1h55m)

The debugging video: why networks train badly even when the code is correct — and I've already met BOTH villains:
- **The 21.9 mystery, solved**: my MLP's initial loss was 21.9 instead of the ignorant 3.29 — confidently-wrong logits at init. This video fixes it (scale down the output layer).
- **Saturated tanh = dying relu's cousin**: units stuck at ±1 have gradient (1−t²)≈0 — my dead-network autopsy, tanh edition. Fix: scale W1 (Kaiming init: gain/sqrt(fan_in)).
- **BatchNorm** — normalize hidden pre-activations per batch; the hack that made deep nets trainable (and couples examples in a batch — watch for why that's weird).
- **Diagnostics** — activation/gradient/update-ratio histograms: how practitioners actually LOOK at a training run.

MY exercises: (1) the init surgery on the starting network, derived not pasted; (2) BatchNorm by hand (mean/std/normalize/gain/bias + running stats); (3) the Linear / BatchNorm1d / Tanh classes (nn.Module's API, hand-built); (4) 6-layer net + read the diagnostic plots.
Boilerplate below: plumbing (5th time) + the four viz cells (Karpathy pastes them too).

Names note: this notebook uses the LECTURE's names (n_embd, n_hidden, block_size, vocab_size) so my code matches the video line-for-line. Same things as my dashboard names.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# BOILERPLATE: data + maps + split (5th time — plumbing)
words = open('names.txt', 'r').read().splitlines()
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)

block_size = 3

def build_dataset(word_list):
    X, Y = [], []
    for w in word_list:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))
Xtr,  Ytr  = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte,  Yte  = build_dataset(words[n2:])
print(Xtr.shape, Xdev.shape, Xte.shape)

## Starting point — the part-2 MLP with NAIVE init (deliberately pre-fix)

The video's first hour is surgery on this cell: watching the initial loss, fixing the output layer's overconfidence, watching tanh saturation, fixing W1's scale, arriving at Kaiming init. **The surgery is my exercise — derive each fix with the video, don't paste the fixed version.**

In [ ]:
# MLP revisited — NAIVE init (the video fixes this step by step; those fixes are MY exercise)
n_embd = 10    # the dimensionality of the character embedding vectors
n_hidden = 200 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647)
C  = torch.randn((vocab_size, n_embd),            generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g)
b1 = torch.randn(n_hidden,                        generator=g)
W2 = torch.randn((n_hidden, vocab_size),          generator=g)
b2 = torch.randn(vocab_size,                      generator=g)

parameters = [C, W1, b1, W2, b2]
print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True

In [ ]:
# BOILERPLATE: the optimization loop (my 5th) — with the lecture's step decay
max_steps = 200000
batch_size = 32
lossi = []

for i in range(max_steps):

    # minibatch construct
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix]

    # forward pass
    emb = C[Xb]
    embcat = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1 + b1
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    lr = 0.1 if i < 100000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    # track stats
    if i % 10000 == 0:
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())

In [ ]:
plt.plot(lossi)

In [ ]:
@torch.no_grad()
def split_loss(split):
    x, y = {
        'train': (Xtr, Ytr),
        'dev':   (Xdev, Ydev),
        'test':  (Xte, Yte),
    }[split]
    emb = C[x]
    embcat = emb.view(emb.shape[0], -1)
    h = torch.tanh(embcat @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, y)
    print(split, loss.item())

split_loss('train')
split_loss('dev')

## EXERCISE ZONE — in video order

1. **Watch the initial loss** (run the loop for a few steps): why is it huge instead of 3.29? Fix the output layer.
2. **Look at `h`**: histogram it, see the saturation at ±1. Why is that bad? (I know why — dying relu, tanh edition.) Fix W1's scale → derive toward Kaiming: gain/sqrt(fan_in), tanh gain = 5/3.
3. **BatchNorm by hand**: normalize `hpreact` per batch (mean/std, keepdim!), then bngain/bnbias, then running mean/std for eval time.
4. **PyTorCHify**: Linear / BatchNorm1d / Tanh classes (same API as nn.Module) — build the 6-layer net.
5. Run the diagnostics below and READ them.

## BOILERPLATE — the four diagnostic plots (verbatim from the lecture)

These run AFTER my Linear/BatchNorm1d/Tanh classes and the deep-net training loop exist (they need `layers`, `parameters`, and `ud` — the update-ratio list the deep loop tracks).

In [ ]:
# visualize histograms
plt.figure(figsize=(20, 4)) # width and height of the plot
legends = []
for i, layer in enumerate(layers[:-1]): # note: exclude the output layer
  if isinstance(layer, Tanh):
    t = layer.out
    print('layer %d (%10s): mean %+.2f, std %.2f, saturated: %.2f%%' % (i, layer.__class__.__name__, t.mean(), t.std(), (t.abs() > 0.97).float().mean()*100))
    hy, hx = torch.histogram(t, density=True)
    plt.plot(hx[:-1].detach(), hy.detach())
    legends.append(f'layer {i} ({layer.__class__.__name__}')
plt.legend(legends);
plt.title('activation distribution')

In [ ]:
# visualize histograms
plt.figure(figsize=(20, 4)) # width and height of the plot
legends = []
for i, layer in enumerate(layers[:-1]): # note: exclude the output layer
  if isinstance(layer, Tanh):
    t = layer.out.grad
    print('layer %d (%10s): mean %+f, std %e' % (i, layer.__class__.__name__, t.mean(), t.std()))
    hy, hx = torch.histogram(t, density=True)
    plt.plot(hx[:-1].detach(), hy.detach())
    legends.append(f'layer {i} ({layer.__class__.__name__}')
plt.legend(legends);
plt.title('gradient distribution')

In [ ]:
# visualize histograms
plt.figure(figsize=(20, 4)) # width and height of the plot
legends = []
for i,p in enumerate(parameters):
  t = p.grad
  if p.ndim == 2:
    print('weight %10s | mean %+f | std %e | grad:data ratio %e' % (tuple(p.shape), t.mean(), t.std(), t.std() / p.std()))
    hy, hx = torch.histogram(t, density=True)
    plt.plot(hx[:-1].detach(), hy.detach())
    legends.append(f'{i} {tuple(p.shape)}')
plt.legend(legends)
plt.title('weights gradient distribution');

In [ ]:
plt.figure(figsize=(20, 4))
legends = []
for i,p in enumerate(parameters):
  if p.ndim == 2:
    plt.plot([ud[j][i] for j in range(len(ud))])
    legends.append('param %d' % i)
plt.plot([0, len(ud)], [-3, -3], 'k') # these ratios should be ~1e-3, indicate on plot
plt.legend(legends);